# Debiasing Combo: sqrt_reweight + oversampling + GroupDRO

This notebook combines multiple debiasing techniques:
- **Pre-processing**: sqrt reweighting by city + oversampling small groups
- **In-processing**: GroupDRO with eta=0.5
- **Training**: 2 epochs for better convergence

In [1]:
# imports

import os
import json
import numpy as np
import pandas as pd
import torch
import joblib
from pathlib import Path

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)
from datasets import Dataset

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# load data

BASE_DIR = Path("..")
PROCESSED_DIR = BASE_DIR / "data" / "processed"

df_train = pd.read_csv(PROCESSED_DIR / "train.csv")
df_val = pd.read_csv(PROCESSED_DIR / "val.csv")
df_test = pd.read_csv(PROCESSED_DIR / "test.csv")

# Load supercategory mapping
mapping = pd.read_csv(PROCESSED_DIR / "label_to_supercategory_v1.csv")
label_to_supercat = dict(zip(mapping["label"], mapping["supercategory"]))

for df in [df_train, df_val, df_test]:
    df["supercategory"] = df["label"].map(label_to_supercat)

print(f"Train: {len(df_train)}, Val: {len(df_val)}, Test: {len(df_test)}")
print(f"\nCity distribution (train):")
print(df_train["city_group"].value_counts().head(10))

Train: 16530, Val: 5510, Test: 5510

City distribution (train):
city_group
Москва             5617
Other              3707
Санкт-Петербург    1797
Краснодар           432
Новосибирск         361
Казань              353
Екатеринбург        292
Самара              271
Ростов-на-Дону      243
Нижний Новгород     235
Name: count, dtype: int64


In [3]:
# label encoding

le = LabelEncoder()
df_train["y"] = le.fit_transform(df_train["supercategory"])
df_val["y"] = le.transform(df_val["supercategory"])
df_test["y"] = le.transform(df_test["supercategory"])

num_labels = len(le.classes_)
print(f"Num labels: {num_labels}")
print(f"Classes: {list(le.classes_)}")

Num labels: 9
Classes: ['backend_general_dev', 'generic_it_ops', 'it_governance_leadership', 'project_product', 'sales_account', 'sysadmin_devops_network', 'tech_support_helpdesk', 'technical_specialized', 'web_frontend']


In [4]:
# PRE-PROCESSING 1: Oversampling small city groups

MIN_CITY_SAMPLES = 100  # Target minimum samples per city
OVERSAMPLE_FACTOR = 2   # How many times to duplicate small groups

city_counts = df_train["city_group"].value_counts()
small_cities = city_counts[city_counts < MIN_CITY_SAMPLES].index.tolist()

print(f"Cities with < {MIN_CITY_SAMPLES} samples: {len(small_cities)}")
print(f"Small cities: {small_cities[:10]}...")

# Oversample small cities
df_train_oversampled = df_train.copy()

for city in small_cities:
    city_samples = df_train[df_train["city_group"] == city]
    # Duplicate samples
    for _ in range(OVERSAMPLE_FACTOR - 1):
        df_train_oversampled = pd.concat([df_train_oversampled, city_samples], ignore_index=True)

print(f"\nOriginal train size: {len(df_train)}")
print(f"After oversampling: {len(df_train_oversampled)}")
print(f"Added samples: {len(df_train_oversampled) - len(df_train)}")

Cities with < 100 samples: 19
Small cities: ['Саратов', 'Барнаул', 'Иркутск', 'Калининград', 'Ижевск', 'Владивосток', 'Тольятти', 'Хабаровск', 'Липецк', 'Калуга']...

Original train size: 16530
After oversampling: 17917
Added samples: 1387


In [5]:
# PRE-PROCESSING 2: Sqrt reweighting by city

# Compute weights on oversampled data
city_counts_new = df_train_oversampled["city_group"].value_counts()

# sqrt inverse frequency weights
raw_w = 1.0 / np.sqrt(city_counts_new)
norm_w = raw_w / raw_w.mean()  # Normalize so mean weight = 1

city_weight_map = norm_w.to_dict()

df_train_oversampled["sample_weight"] = df_train_oversampled["city_group"].map(city_weight_map).astype(float)

print("Sample weight stats:")
print(df_train_oversampled["sample_weight"].describe())

print("\nTop-5 highest weights (rarest cities):")
print(pd.Series(city_weight_map).sort_values(ascending=False).head())

print("\nTop-5 lowest weights (most common cities):")
print(pd.Series(city_weight_map).sort_values(ascending=True).head())

Sample weight stats:
count    17917.000000
mean         0.510290
std          0.395720
min          0.180114
25%          0.180114
50%          0.221711
75%          0.905988
max          1.304990
Name: sample_weight, dtype: float64

Top-5 highest weights (rarest cities):
Томск               1.304990
Ярославль           1.298935
Набережные Челны    1.264290
Волгоград           1.264290
Кемерово            1.242677
dtype: float64

Top-5 lowest weights (most common cities):
Москва             0.180114
Other              0.221711
Санкт-Петербург    0.318438
Краснодар          0.649467
Новосибирск        0.710470
dtype: float64


In [6]:
# create city id for GDRO

city_to_id = {c: i for i, c in enumerate(df_train_oversampled["city_group"].unique())}
num_groups = len(city_to_id)

df_train_oversampled["city_id"] = df_train_oversampled["city_group"].map(city_to_id).astype(int)
df_val["city_id"] = df_val["city_group"].map(city_to_id).fillna(-1).astype(int)
df_test["city_id"] = df_test["city_group"].map(city_to_id).fillna(-1).astype(int)

print(f"Number of city groups for GroupDRO: {num_groups}")

Number of city groups for GroupDRO: 41


In [7]:
# tokenizer

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(
        batch["resume_text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

In [8]:
# build datasets

train_dataset = Dataset.from_pandas(
    df_train_oversampled[["resume_text", "y", "city_id", "sample_weight"]]
)
val_dataset = Dataset.from_pandas(
    df_val[["resume_text", "y", "city_id"]]
)
test_dataset = Dataset.from_pandas(
    df_test[["resume_text", "y", "city_id"]]
)

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset = train_dataset.rename_column("y", "labels")
val_dataset = val_dataset.rename_column("y", "labels")
test_dataset = test_dataset.rename_column("y", "labels")

train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels", "city_id", "sample_weight"])
val_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels", "city_id"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels", "city_id"])

print(f"Train dataset size: {len(train_dataset)}")
print(f"Train sample keys: {train_dataset[0].keys()}")

Map: 100%|██████████| 5510/5510 [00:08<00:00, 680.17 examples/s]

Train dataset size: 17917
Train sample keys: dict_keys(['labels', 'city_id', 'sample_weight', 'input_ids', 'attention_mask'])


In [9]:
# groupDRO + weighted trainer

class GroupDROWeightedTrainer(Trainer):
    
    def __init__(self, *args, num_groups: int, eta: float = 0.5, **kwargs):
        super().__init__(*args, **kwargs)
        self.num_groups = num_groups
        self.eta = eta
        # Initialize group weights uniformly
        self.q = torch.ones(num_groups) / num_groups
        print(f"GroupDRO initialized: {num_groups} groups, eta={eta}")

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None, **kwargs):
        # Extract extra fields
        city_id = inputs.pop("city_id")
        sample_weight = inputs.pop("sample_weight", None)
        labels = inputs.get("labels")

        # Forward pass
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # Per-sample cross-entropy loss
        loss_fct = torch.nn.CrossEntropyLoss(reduction="none")
        per_sample_loss = loss_fct(logits, labels)

        # Apply sample weights (sqrt reweighting)
        if sample_weight is not None:
            sample_weight = sample_weight.to(per_sample_loss.dtype)
            per_sample_loss = per_sample_loss * sample_weight

        # Compute per-group loss for GroupDRO
        device = per_sample_loss.device
        group_loss = torch.zeros(self.num_groups, device=device)
        group_count = torch.zeros(self.num_groups, device=device)

        for g in range(self.num_groups):
            mask = (city_id == g)
            if mask.any():
                group_loss[g] = per_sample_loss[mask].mean()
                group_count[g] = mask.sum().float()

        # Update group weights (exponentiated gradient)
        with torch.no_grad():
            q = self.q.to(device)
            # Only update for groups present in this batch
            group_present = (group_count > 0).float()
            q = q * torch.exp(self.eta * group_loss * group_present)
            q = q / q.sum()  # Normalize
            self.q = q.detach().cpu()

        # Final loss: weighted sum of group losses
        q = self.q.to(device)
        loss = (q * group_loss).sum()

        return (loss, outputs) if return_outputs else loss

In [10]:
# training args

MODEL_NAME = "bert_debiased_combo"

training_args = TrainingArguments(
    output_dir=f"./models/{MODEL_NAME}",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,  # 2 epochs like the best model
    weight_decay=0.01,
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    remove_unused_columns=False,  # Important for custom fields
)

print(f"Training config:")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Learning rate: {training_args.learning_rate}")

Training config:
  Epochs: 2
  Batch size: 8
  Learning rate: 2e-05


In [11]:
# metrics

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro")
    }

In [12]:
# initialize model

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)

print(f"Model: {model_name}")
print(f"Num labels: {num_labels}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1603.41it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those p

Model: bert-base-uncased
Num labels: 9


In [13]:
# train!

ETA = 0.5  # step size

trainer = GroupDROWeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    num_groups=num_groups,
    eta=ETA
)

print(f"\n{'='*60}")
print(f"Starting training...")
print(f"  Method: sqrt_reweight + oversampling + GroupDRO")
print(f"  GroupDRO eta: {ETA}")
print(f"  Num groups: {num_groups}")
print(f"{'='*60}\n")

trainer.train()

GroupDRO initialized: 41 groups, eta=0.5

Starting training...
  Method: sqrt_reweight + oversampling + GroupDRO
  GroupDRO eta: 0.5
  Num groups: 41



/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.293337,1.499524,0.393648,0.313030
2,0.280195,1.277584,0.493466,0.494890


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.04it/s]
/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.41it/s]
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.La

TrainOutput(global_step=4480, training_loss=0.3317154083933149, metrics={'train_runtime': 5023.0192, 'train_samples_per_second': 7.134, 'train_steps_per_second': 0.892, 'total_flos': 2357228532358656.0, 'train_loss': 0.3317154083933149, 'epoch': 2.0})

In [14]:
# evaluate on test set

predictions = trainer.predict(test_dataset)

y_true = predictions.label_ids
y_pred = np.argmax(predictions.predictions, axis=1)

print(f"\n{'='*60}")
print("TEST RESULTS")
print(f"{'='*60}")
print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
print(f"Macro F1: {f1_score(y_true, y_pred, average='macro'):.4f}")

/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)



TEST RESULTS
Accuracy: 0.4882
Macro F1: 0.4808


In [15]:
# fairness eval

df_eval = df_test.copy()
df_eval["y_true"] = y_true
df_eval["y_pred"] = y_pred
df_eval["correct"] = df_eval["y_true"] == df_eval["y_pred"]

def gap_by_group(df, group_col):
    group_acc = df.groupby(group_col)["correct"].mean().dropna()
    return group_acc.max() - group_acc.min() if len(group_acc) > 0 else np.nan

def ovr_rates_by_group(df, group_col, num_classes):
    groups = sorted(df[group_col].dropna().unique())
    tpr = np.zeros((len(groups), num_classes))
    support = np.zeros((len(groups), num_classes))
    
    for gi, g in enumerate(groups):
        df_g = df[df[group_col] == g]
        yt = df_g["y_true"].values
        yp = df_g["y_pred"].values
        
        for c in range(num_classes):
            pos_mask = (yt == c)
            TP = np.sum((yp == c) & pos_mask)
            FN = np.sum((yp != c) & pos_mask)
            support[gi, c] = np.sum(pos_mask)
            tpr[gi, c] = TP / (TP + FN) if (TP + FN) > 0 else np.nan
    
    return tpr, support, groups

def summarize_gaps(tpr, support=None, min_support=0):
    gaps = []
    for c in range(tpr.shape[1]):
        col = tpr[:, c]
        if support is not None and min_support > 0:
            mask = support[:, c] >= min_support
            col = col[mask]
        col = col[~np.isnan(col)]
        gaps.append(np.max(col) - np.min(col) if len(col) >= 2 else np.nan)
    
    gaps = np.array(gaps)
    valid = gaps[~np.isnan(gaps)]
    return np.max(valid) if len(valid) > 0 else np.nan, np.mean(valid) if len(valid) > 0 else np.nan

# Compute metrics
tpr, support, groups = ovr_rates_by_group(df_eval, "city_group", num_labels)

tpr_worst_all, tpr_macro_all = summarize_gaps(tpr)
tpr_worst_robust, tpr_macro_robust = summarize_gaps(tpr, support, min_support=30)

print(f"\n{'='*60}")
print("FAIRNESS METRICS")
print(f"{'='*60}")
print(f"City Accuracy Gap: {gap_by_group(df_eval, 'city_group'):.4f}")
print(f"")
print(f"TPR Gap (all cities):")
print(f"  Worst: {tpr_worst_all:.4f}")
print(f"  Macro: {tpr_macro_all:.4f}")
print(f"")
print(f"TPR Gap (robust, support >= 30):")
print(f"  Worst: {tpr_worst_robust:.4f}")
print(f"  Macro: {tpr_macro_robust:.4f}")


FAIRNESS METRICS
City Accuracy Gap: 0.5858

TPR Gap (all cities):
  Worst: 1.0000
  Macro: 0.9475

TPR Gap (robust, support >= 30):
  Worst: 0.4059
  Macro: 0.1378


In [16]:
# save model

SAVE_DIR = Path("models") / MODEL_NAME
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# save model and tokenizer
model.save_pretrained(SAVE_DIR, safe_serialization=True)
tokenizer.save_pretrained(SAVE_DIR)

# Save label mapping
classes = list(le.classes_)
label2id = {label: int(i) for i, label in enumerate(classes)}
id2label = {int(i): label for i, label in enumerate(classes)}

with open(SAVE_DIR / "label_mapping.json", "w", encoding="utf-8") as f:
    json.dump({"label2id": label2id, "id2label": id2label}, f, ensure_ascii=False, indent=2)

# Save label encoder
joblib.dump(le, SAVE_DIR / "label_encoder.joblib")

# Save training config
config = {
    "method": "sqrt_reweight + oversampling + GroupDRO",
    "eta": ETA,
    "num_groups": num_groups,
    "oversample_factor": OVERSAMPLE_FACTOR,
    "min_city_samples": MIN_CITY_SAMPLES,
    "epochs": training_args.num_train_epochs,
    "accuracy": float(accuracy_score(y_true, y_pred)),
    "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
    "tpr_gap_worst_robust": float(tpr_worst_robust),
    "tpr_gap_macro_robust": float(tpr_macro_robust)
}

with open(SAVE_DIR / "training_config.json", "w") as f:
    json.dump(config, f, indent=2)

print(f"\nModel saved to: {SAVE_DIR}")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.26it/s]


Model saved to: models/bert_debiased_combo


In [17]:
# summary

print(f"\n{'='*60}")
print("TRAINING COMPLETE")
print(f"{'='*60}")
print(f"")
print(f"Method: sqrt_reweight + oversampling + GroupDRO (eta={ETA})")
print(f"")
print(f"Results:")
print(f"  Accuracy:              {accuracy_score(y_true, y_pred):.4f}")
print(f"  Macro F1:              {f1_score(y_true, y_pred, average='macro'):.4f}")
print(f"  TPR Gap worst (robust): {tpr_worst_robust:.4f}")
print(f"  TPR Gap macro (robust): {tpr_macro_robust:.4f}")
print(f"")
print(f"Compare with current best (BERT 2ep sqrt_rw):")
print(f"  Accuracy:              0.609")
print(f"  TPR Gap worst (robust): 0.329")
print(f"  TPR Gap macro (robust): 0.116")


TRAINING COMPLETE

Method: sqrt_reweight + oversampling + GroupDRO (eta=0.5)

Results:
  Accuracy:              0.4882
  Macro F1:              0.4808
  TPR Gap worst (robust): 0.4059
  TPR Gap macro (robust): 0.1378

Compare with current best (BERT 2ep sqrt_rw):
  Accuracy:              0.609
  TPR Gap worst (robust): 0.329
  TPR Gap macro (robust): 0.116
